# QueueLab development pilot
This notebook runs the same CLI as the local quickstart. It downloads a base model to the cloud runtime, not your laptop. The synthetic cases are unreviewed development cases. No LoRA training is included yet.

Set `REPO_REF` to a published release tag or commit to pin the code and fixtures. Start with one opening. Set `OPENINGS = 10` for the full pilot. If you enable a Colab GPU runtime, also set `DEVICE = 'cuda'`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_REF = 'main'
REPO = Path('/content/queuelab')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/assafbar2/queuelab.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)

In [ ]:
from uuid import uuid4

reference = f'reports/reference-{uuid4().hex[:8]}.json'
subprocess.run([sys.executable, '-m', 'harness', 'run', '--backend', 'rules', '--output', reference], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'harness', 'rescore', reference], cwd=REPO, check=True)

## Optional: check one Falcon action
This asks Falcon for one tool request and prints its raw reply. It does not read a record or run a ticket. The first model cell downloads Falcon to Colab.

In [ ]:
subprocess.run(['uv', 'run', '--locked', '--extra', 'model', 'python', '-m', 'harness', 'probe-falcon'], cwd=REPO, check=True)

In [ ]:
from google.colab import files

DEVICE = 'cpu'
OPENINGS = 1
baseline = f'reports/falcon-{uuid4().hex[:8]}.json'
command = ['uv', 'run', '--locked', '--extra', 'model', 'python', '-m', 'harness']
subprocess.run([*command, 'run', '--backend', 'falcon', '--device', DEVICE, '--limit-openings', str(OPENINGS), '--output', baseline], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'harness', 'rescore', baseline], cwd=REPO, check=True)
files.download(str(REPO / baseline))